# BONITA — extrinsic evaluation on BoolQ (Italian)

*Francesco Baiocchi, Leonardo Petrilli — Task 3, course project work*

**Contents**

- [Methodology](#methodology)
- [1. Setup & configuration](#1-setup--configuration)
- [2. Data: BoolQ splits and BONITA meta-template](#2-data-boolq-splits-and-bonita-meta-template)
- [3. Training BONITA](#3-training-bonita)
- [4. Unannotated Italian text](#4-unannotated-italian-text)
- [5. Generating the instruction-tuning dataset with BONITA](#5-generating-the-instruction-tuning-dataset-with-bonita)
- [6. Training the student](#6-training-the-student)
- [7. Log-likelihood evaluation on the held-out 20%](#7-log-likelihood-evaluation-on-the-held-out-20)
- [8. Results](#8-results)

## Methodology

**Goal.** BONITA is an Italian version of Bonito ([Nayak et al., 2024](https://arxiv.org/abs/2402.18334)): a model that turns unannotated text into instruction-tuning data. Given a passage and a task type, it generates a task, i.e. an instruction with its input and the expected answer. Here the task type is boolean question answering: from a passage, BONITA writes a yes/no question about it and the answer (`Vero`/`Falso`).

**Pipeline.**

1. **Train BONITA**: a QLoRA adapter (plus the embeddings of its special tokens) on [`sapienzanlp/Minerva-7B-base-v1.0`](https://huggingface.co/sapienzanlp/Minerva-7B-base-v1.0), using 80% of [`sapienzanlp/boolq_italian`](https://huggingface.co/datasets/sapienzanlp/boolq_italian) (`train` + `validation`, split by passage, see Section 2).
2. **Collect unannotated Italian text**: 10,000 paragraphs from Italian Wikipedia, the same kind of source as BoolQ, whose passages are English Wikipedia paragraphs translated into Italian. No label is attached to them.
3. **Generate a synthetic dataset**: BONITA writes one question/answer pair per paragraph. Generations that cannot be parsed and duplicate questions are discarded.
4. **Train the student**: a QLoRA adapter on [`sapienzanlp/Minerva-7B-instruct-v1.0`](https://huggingface.co/sapienzanlp/Minerva-7B-instruct-v1.0), trained on the synthetic dataset. We call it student because it learns the task only from data written by BONITA.
5. **Evaluate** the baseline model (Minerva-7B-instruct without the adapter) and the student on the remaining 20% (2,490 human-labelled examples). Test examples whose passage or question also appears in BONITA's data are removed, so the test set is never seen by BONITA. Each example is scored by log-likelihood, comparing `Vero` and `Falso`, with 3 instruction variants; we report accuracy and macro-F1 per variant and averaged, next to the majority baseline (always `Vero`).

## 1. Setup & configuration

The whole pipeline runs on a single NVIDIA GeForce RTX 3090 (24 GB). Models are handled with [`transformers`](https://github.com/huggingface/transformers) (loading, generation), [`bitsandbytes`](https://github.com/bitsandbytes-foundation/bitsandbytes) (4-bit quantization), [`peft`](https://github.com/huggingface/peft) (LoRA adapters) and [`trl`](https://github.com/huggingface/trl) (`SFTTrainer`, supervised fine-tuning on prompt/completion pairs); datasets are loaded from the Hugging Face Hub with [`datasets`](https://github.com/huggingface/datasets).

In [1]:
import os

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

import gc
import json
import random
import re
import tempfile

import numpy as np
import pandas as pd
import torch
import transformers
from datasets import Dataset, concatenate_datasets, load_dataset
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, EarlyStoppingCallback
from trl.trainer.sft_config import SFTConfig
from trl.trainer.sft_trainer import SFTTrainer

In [2]:
SEED = 42

BASE_MODEL = "sapienzanlp/Minerva-7B-base-v1.0"
INSTRUCTED_MODEL = "sapienzanlp/Minerva-7B-instruct-v1.0"

# outputs of each stage (each stage can be re-run from its files)
BONITA_DIR = "ckpt/bonita_full"  # generator adapter + tokenizer
WIKI_PATH = "wiki_passages.jsonl"  # unannotated passages
GENERATED_PATH = "generated_tasks.jsonl"  # parsed and filtered BONITA tasks
STUDENT_DIR = "ckpt/student_bonita"  # student adapter
RESULTS_PATH = "results.json"  # generation statistics and metrics

TEST_FRAC = 0.2  # share of BoolQ-Italian passages held out as the final test set of the baseline and the student
N_PASSAGES = 10_000  # unannotated paragraphs given to BONITA (~ size of BoolQ train)

# one of the 10 files of Italian Wikipedia (~180k articles) is enough to sample the passages
WIKI_SHARD = "hf://datasets/wikimedia/wikipedia/20231101.it/train-00000-of-00010.parquet"

# for both, the checkpoint with the lowest loss on a held-out 5% of the training passages is kept
# (evaluated every 100 steps, early stopping after 3 evaluations without improvement)
BONITA_EPOCHS = 4
STUDENT_EPOCHS = 4
DEV_FRAC = 0.05


def free_memory() -> None:
    gc.collect()
    torch.cuda.empty_cache()


def read_jsonl(path: str) -> list[dict]:
    with open(path) as f:
        return [json.loads(line) for line in f]


def write_jsonl(rows: list[dict], path: str) -> None:
    with open(path, "w") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


transformers.set_seed(SEED)  # seeds random, numpy and torch (CPU and CUDA)

The next cells define the components shared by all the models of the pipeline.

**Quantization.** Every model is loaded in 4-bit NF4, with double quantization and `bfloat16` compute. This includes the baseline, Minerva-7B-instruct, which is evaluated in the same quantized form as the student: the two only differ by the adapter.

In [3]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)


def load_model(tokenizer=None, model_name: str = BASE_MODEL) -> transformers.PreTrainedModel:
    """4-bit model; the embeddings are resized only if `tokenizer` has no free rows left for its added tokens."""
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
        dtype=torch.bfloat16,
    )
    if tokenizer is not None and len(tokenizer) > model.config.vocab_size:
        model.resize_token_embeddings(len(tokenizer))
    return model

**LoRA.** BONITA and the student are trained with the same LoRA configuration, applied to all linear projections with $r=16$, $\alpha=32$ and dropout $0.05$. BONITA additionally trains the embedding and LM-head rows of its special tokens.

In [ ]:
def lora_config(trainable_token_ids: list[int] | None = None) -> LoraConfig:
    """LoRA on all linear projections, plus the rows of `trainable_token_ids` in the (untied) embeddings and LM head."""
    return LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules="all-linear",
        lora_dropout=0.05,
        task_type="CAUSAL_LM",
        trainable_token_indices={"embed_tokens": trainable_token_ids, "lm_head": trainable_token_ids}
        if trainable_token_ids
        else None,
    )

**Training.** Both models share the same arguments (`sft_config`): a batch of 8 with 2 accumulation steps (effective batch 16), a learning rate of $5\times10^{-5}$ with cosine decay and 3% warmup, `bfloat16` and at most 4 epochs. The loss is computed on the completion only, so the prompt is read as input but is never a training target. Every 100 steps the model is evaluated on a dev set made of 5% of the training passages; training stops early if the dev loss does not improve for 3 evaluations in a row, and the checkpoint with the lowest dev loss is restored at the end.

In [ ]:
def sft_config(output_dir: str, num_epochs: int) -> SFTConfig:
    """Training arguments shared by BONITA and the student; the checkpoint with the lowest dev loss is kept."""
    return SFTConfig(
        output_dir=output_dir,
        num_train_epochs=num_epochs,
        per_device_train_batch_size=8,
        gradient_accumulation_steps=2,
        learning_rate=5e-5,
        lr_scheduler_type="cosine",
        warmup_steps=0.03,  # float in [0, 1) = ratio of total steps
        bf16=True,
        completion_only_loss=True,
        eval_strategy="steps",  # dev loss every 100 steps
        eval_steps=100,
        save_steps=100,
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        logging_steps=25,
        seed=SEED,
    )


def train_lora(
    model,
    tokenizer,
    train_rows: list[dict],
    dev_rows: list[dict],
    output_dir: str,
    num_epochs: int,
    trainable_token_ids: list[int] | None = None,
) -> None:
    """Adds a fresh LoRA to `model`, trains it, saves the best adapter to `output_dir` and prints the dev losses."""
    transformers.set_seed(SEED)  # the LoRA weights are initialized before the Trainer sets its own seed
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)  # freeze, fp32 norms, grad ckpt
    model = get_peft_model(model, lora_config(trainable_token_ids))
    assert isinstance(model, PeftModel)  # not a PeftMixedModel (mixed=False)
    model.print_trainable_parameters()
    with tempfile.TemporaryDirectory() as ckpt_dir:  # intermediate checkpoints, deleted after training
        trainer = SFTTrainer(
            model=model,
            args=sft_config(ckpt_dir, num_epochs),
            train_dataset=Dataset.from_list(train_rows).select_columns(["prompt", "completion"]),
            eval_dataset=Dataset.from_list(dev_rows).select_columns(["prompt", "completion"]),
            processing_class=tokenizer,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],  # stop after 3 evals without improvement
        )
        trainer.train()
    # the best checkpoint is loaded into `model`; only the LoRA and the trained token rows are saved
    # (PEFT would otherwise save the whole resized embeddings and LM head, ~1.7 GB)
    model.save_pretrained(output_dir, save_embedding_layers=False)
    tokenizer.save_pretrained(output_dir)
    print(pd.DataFrame(trainer.state.log_history).dropna(subset=["eval_loss"])[["step", "epoch", "eval_loss"]])

## 2. Data: BoolQ splits and BONITA meta-template

[BoolQ-Italian](https://huggingface.co/datasets/sapienzanlp/boolq_italian) is an Italian translation of [BoolQ](https://huggingface.co/datasets/google/boolq), a question-answering dataset composed of user queries issued to a search engine. The task is to predict whether the answer to the question is true or false based on the context provided in the question.

In [20]:
boolq = load_dataset("sapienzanlp/boolq_italian")
print(f"train examples: {len(boolq['train'])}")
print(f"validation examples: {len(boolq['validation'])}")

train examples: 9399
validation examples: 3259


**Pre-processing.** While manually inspecting the dataset, we noticed that some passages were duplicated across the splits. This comes from the original BoolQ, whose splits are drawn per question: different user queries, often paraphrases, were paired with the same Wikipedia paragraph.

To avoid data leakage we merge the two splits and divide them (80/20) by English passage, putting all the questions on a paragraph on the same side, so no passage is shared between BONITA's training data and the held-out set.

In [21]:
# passages that appear in both the original splits
train_passages = {m["passage_translation"] for m in boolq["train"]["metadata"]}
validation_passages = {m["passage_translation"] for m in boolq["validation"]["metadata"]}
print(f"passages in both train and validation: {len(train_passages & validation_passages)}")

passages in both train and validation: 487


In [22]:
def split_by_group(rows: list[dict], key: str, frac: float, seed: int) -> tuple[list[dict], list[dict]]:
    """Splits rows into (train, held-out) so that all rows sharing `key` end up on the same side."""
    groups = sorted({r[key] for r in rows})
    random.Random(seed).shuffle(groups)
    held_out = set(groups[: round(len(groups) * frac)])
    return [r for r in rows if r[key] not in held_out], [r for r in rows if r[key] in held_out]


boolq_all = [
    {**ex, "passage": ex["metadata"]["passage"]}
    for ex in concatenate_datasets([boolq["train"], boolq["validation"]]).to_list()
]

bonita_source, boolq_test = split_by_group(
    boolq_all, "passage", frac=TEST_FRAC, seed=SEED
)  # by original (English) passage
print(f"BONITA source: {len(bonita_source)}, held out: {len(boolq_test)}")

BONITA source: 10094, held out: 2564


Splitting by English passage is not enough to keep the test set clean. Both cases below already exist in the original BoolQ, and the translation makes them identical in Italian:

- two English passages can be near-identical revisions of the same Wikipedia paragraph (e.g. differing only by a corrected typo), and they get the same Italian translation;
- BoolQ questions are real search queries, so different users ask the same thing in different words (*"Is there a second season of Iron Fist?"* / *"Is there going to be a Iron Fist season 2?"*), and the paraphrases get the same Italian translation.

We therefore drop every held-out example whose Italian passage, or whose normalized question (lowercased, punctuation removed), also appears in BONITA's data. This removes 74 examples (31 by passage, 45 by question, 2 by both) and leaves a test set of 2,490 examples that BONITA has never seen, not even in another form.

In [ ]:
def normalize(question: str) -> str:
    return " ".join(re.findall(r"\w+", question.lower()))


# held-out examples whose Italian passage or normalized question also appear in BONITA's data
seen_passages = {ex["metadata"]["passage_translation"] for ex in bonita_source}
seen_questions = {normalize(ex["input_translation"]) for ex in bonita_source}


def is_near_duplicate(ex: dict) -> bool:
    passage, question = ex["metadata"]["passage_translation"], normalize(ex["input_translation"])
    return passage in seen_passages or question in seen_questions


print(f"near-duplicates of BONITA's data in the held-out set: {sum(is_near_duplicate(ex) for ex in boolq_test)}")

boolq_test = [ex for ex in boolq_test if not is_near_duplicate(ex)]

train_passages = {m["metadata"]["passage_translation"] for m in bonita_source}
test_passages = {m["metadata"]["passage_translation"] for m in boolq_test}
print(f"passages in both train and validation afer splitting: {len(train_passages & test_passages)}")
print(f"near-duplicates after filtering: {sum(is_near_duplicate(ex) for ex in boolq_test)}")

BONITA source: 10094, held out: 2564
near-duplicates of BONITA's data in the held-out set: 74
passages in both train and validation afer splitting: 0
near-duplicates after filtering: 0


Each BoolQ row becomes a prompt/completion pair:

**Prompt** (input):
```
<|tasktype|>
boolean question answering
<|context|>
{passage}
<|task|>
```

**Completion** (target):
```
{instruction} {question} Vero o Falso?
<|pipe|>
Risposta: Vero | Falso
```

The prompt contains the task type and the passage; the completion is the whole task, i.e. the instruction with the question, and after `<|pipe|>` the answer. BONITA therefore learns to generate both the question and its answer from a passage. The instruction is drawn at random from three equivalent Italian wordings, so that the generator does not always produce the same one. A 5% slice of BONITA's passages is kept aside to monitor its dev loss.

In [14]:
INSTRUCTION_VARIANTS = [
    "Dopo aver letto il passaggio, rispondi alla seguente domanda:",
    "In base al testo precedente, rispondi alla domanda:",
    "Secondo quanto riportato nel passaggio, rispondi alla domanda:",
]
QUESTION_SUFFIX = "Vero o Falso?"
SPECIAL_TOKENS = ["<|tasktype|>", "<|context|>", "<|task|>", "<|pipe|>"]

BONITA_INPUT = "<|tasktype|>\nboolean question answering\n<|context|>\n{passage}\n<|task|>"
BONITA_OUTPUT = "\n{instruction} {question}\n<|pipe|>\n{answer}"
PROMPT_TEMPLATE = "{passage}\n\n{task}\nRisposta:"  # student training and evaluation (Sections 6-7)


def answer_text(label: bool) -> str:
    return "Vero" if label else "Falso"


rng = random.Random(SEED)
bonita_rows = [
    {
        "prompt": BONITA_INPUT.format(passage=ex["metadata"]["passage_translation"]),
        "completion": BONITA_OUTPUT.format(
            instruction=rng.choice(INSTRUCTION_VARIANTS),
            question=f"{ex['input_translation']} {QUESTION_SUFFIX}",
            answer=f"Risposta: {answer_text(ex['label'])}",
        ),
        "passage": ex["passage"],
    }
    for ex in bonita_source
]
bonita_train, bonita_dev = split_by_group(bonita_rows, "passage", frac=DEV_FRAC, seed=SEED)
print(f"BONITA train: {len(bonita_train)}, dev: {len(bonita_dev)}")
print(bonita_train[0]["prompt"] + bonita_train[0]["completion"])

BONITA train: 9611, dev: 483
<|tasktype|>
boolean question answering
<|context|>
Il persiano (/ˈpɜːrʒən, -ʃən/), conosciuto anche con il suo nome endonimo Farsi (فارسی fārsi (fɒːɾˈsiː) (ascolta)), è una delle lingue iraniche occidentali del ramo indo-iraniano della famiglia delle lingue indoeuropee. È parlato principalmente in Iran, Afghanistan (ufficialmente conosciuto come Dari dal 1958) e Tagikistan (ufficialmente conosciuto come Tajiki dall'era sovietica) e in alcune altre regioni che storicamente erano società persiane e considerate parte del Grande Iran. È scritto in alfabeto persiano, una variante modificata della scrittura araba, che a sua volta si è evoluta dall'alfabeto aramaico.
<|task|>
Secondo quanto riportato nel passaggio, rispondi alla domanda: L'Iran e l'Afghanistan parlano la stessa lingua? Vero o Falso?
<|pipe|>
Risposta: Vero


## 3. Training BONITA

The four markers of the meta-template (`<|tasktype|>`, `<|context|>`, `<|task|>`, `<|pipe|>`) are added to the tokenizer as special tokens, so that each one is a single, unsplittable token. The embedding matrix is resized only if it has no free rows left for them, which is the case for Minerva-7B-base (exactly 51,200 rows).

> **Note:** this section is **not meant to be re-run**. Training BONITA takes several GPU-hours: to train it again, use the script `scripts/train_bonita.py`. The run is documented here for transparency and as a reference. The trained adapter and tokenizer are provided in `ckpt/bonita_full/`, from where Section 5 loads them: the next cell can be skipped.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.add_tokens(SPECIAL_TOKENS, special_tokens=True)
model = load_model(tokenizer)
special_token_ids = tokenizer.convert_tokens_to_ids(SPECIAL_TOKENS)
assert isinstance(special_token_ids, list)  # a list of tokens gives a list of ids

# LoRA + the 4 rows of the special tokens
train_lora(model, tokenizer, bonita_train, bonita_dev, BONITA_DIR, BONITA_EPOCHS, trainable_token_ids=special_token_ids)

del model
free_memory()

## 4. Unannotated Italian text

[BoolQ](https://arxiv.org/abs/1905.10044) passages are Wikipedia paragraphs, so the unannotated text is taken from Italian Wikipedia ([`wikimedia/wikipedia`](https://huggingface.co/datasets/wikimedia/wikipedia), `20231101.it`). Only the first of its 10 parquet files is downloaded (~180k articles, far more than needed) and its articles are shuffled. From each article we take one paragraph of prose (ending with a full stop and at least as long as the 5th percentile of the length of BONITA's training passages in words, so that it is not much shorter than BoolQ passages), chosen at random.

The passages are collected only once and saved to `wiki_passages.jsonl`; if the file already exists it is loaded instead. To collect them again, delete the file.

In [ ]:
MIN_WORDS = int(np.percentile([len(ex["metadata"]["passage_translation"].split()) for ex in bonita_source], 5))


def candidate_paragraphs(text: str) -> list[str]:
    paragraphs = (p.strip() for p in text.split("\n"))
    return [p for p in paragraphs if p.endswith(".") and len(p.split()) >= MIN_WORDS]


def collect_wiki_passages() -> list[dict]:
    wiki = load_dataset("parquet", data_files=WIKI_SHARD, split="train")
    assert isinstance(wiki, Dataset)  # a single split, not a DatasetDict
    wiki = wiki.shuffle(seed=SEED)
    rng = random.Random(SEED)
    passages = []
    for text in wiki["text"]:  # lazy column: one article at a time
        candidates = candidate_paragraphs(text)
        if candidates:
            passages.append({"passage": rng.choice(candidates)})
        if len(passages) >= N_PASSAGES:
            break
    return passages


# the passages are collected once
if os.path.exists(WIKI_PATH):
    print(f"loading the passages from {WIKI_PATH}")
    wiki_passages = read_jsonl(WIKI_PATH)
else:
    print(f"minimum paragraph length: {MIN_WORDS} words")
    wiki_passages = collect_wiki_passages()
    write_jsonl(wiki_passages, WIKI_PATH)

print(f"{len(wiki_passages)} passages")
print(wiki_passages[0])

## 5. Generating the instruction-tuning dataset with BONITA

BONITA generates one task per passage with greedy decoding (`do_sample=False`, for reproducibility); the other generation parameters are the defaults of Minerva-7B-base. Each generation is parsed into instruction + question and answer, and kept only if:

- it contains a parseable `Risposta: Vero/Falso` and the question ends with *"Vero o Falso?"*;
- the question is not a duplicate of an already kept question (after normalization).

The next cell reloads the base model with the BONITA adapter and generates in batches of 16 prompts, each made of the task type and a Wikipedia passage (the prompt part of the meta-template). The prompts of a batch have different lengths and are padded **on the left**: in this way every prompt ends right where generation starts, and the new tokens can be taken as everything after the input.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BONITA_DIR)
model = PeftModel.from_pretrained(load_model(tokenizer), BONITA_DIR)
model.eval()


@torch.no_grad()
def batch_generate(model, tokenizer, prompts: list[str], batch_size: int = 16, **generation_kwargs) -> list[str]:
    """Generates one completion per prompt (new tokens only, special tokens removed)."""
    predictions = []
    for i in tqdm(range(0, len(prompts), batch_size)):
        # left padding: every prompt ends right before the new tokens
        inputs = tokenizer(prompts[i : i + batch_size], padding=True, padding_side="left", return_tensors="pt").to(
            model.device
        )
        outputs = model.generate(
            **inputs, pad_token_id=tokenizer.pad_token_id, tokenizer=tokenizer, **generation_kwargs
        )
        new_tokens = outputs[:, inputs["input_ids"].shape[1] :]
        predictions += tokenizer.batch_decode(new_tokens, skip_special_tokens=True)
    return predictions


generations = batch_generate(
    model,
    tokenizer,
    [BONITA_INPUT.format(passage=p["passage"]) for p in wiki_passages],
    max_new_tokens=256,
    stop_strings=["Risposta: Vero", "Risposta: Falso"],
    do_sample=False,
    use_cache=True,  # Minerva's config disables the KV cache
)

del model
free_memory()

**Parsing and filtering.** A generation is expected to look like `{instruction} {question} Vero o Falso? Risposta: Vero|Falso` (the `<|pipe|>` token is removed when decoding). The label is found with a regular expression on `Risposta: Vero/Falso`; everything before it is the *task* (instruction + question), which is what the student will see. The bare question (task without instruction and suffix) is only used for deduplication, after lowercasing and removing punctuation.

The kept tasks (passage, task, question, label) are saved to `generated_tasks.jsonl`. The counts of malformed and duplicate generations and the share of `Vero` labels, which shows whether BONITA's answers are balanced, are saved to `results.json` in Section 8.

In [8]:
LABEL_RE = re.compile(r"Risposta:\s*(Vero|Falso)\b", re.IGNORECASE)


def parse_generation(text: str) -> dict | None:
    """Splits a BONITA generation into task (instruction + question), question and label."""
    match = LABEL_RE.search(text)
    if match is None:
        return None
    task = text[: match.start()].strip()
    if not task.endswith(QUESTION_SUFFIX):
        return None
    question = task.removesuffix(QUESTION_SUFFIX)
    for variant in INSTRUCTION_VARIANTS:
        question = question.replace(variant, "")
    return {"task": task, "question": question.strip(), "label": match.group(1).capitalize() == "Vero"}


generated, seen = [], set()
counts: dict[str, float] = {"generated": len(generations), "malformed": 0, "duplicate": 0}
for passage, text in zip(wiki_passages, generations, strict=True):
    parsed = parse_generation(text)
    if parsed is None:
        counts["malformed"] += 1
        continue
    key = normalize(parsed["question"])
    if key in seen:
        counts["duplicate"] += 1
        continue
    seen.add(key)
    generated.append({"passage": passage["passage"], **parsed})

write_jsonl(generated, GENERATED_PATH)

counts["kept"] = len(generated)
counts["vero_rate"] = sum(r["label"] for r in generated) / len(generated) if generated else 0.0
print(counts)
print(generated[0]["passage"], "...")
print("  ->", generated[0]["task"], "|", answer_text(generated[0]["label"]))

loading the generated tasks from generated_tasks.jsonl


NameError: name 'counts' is not defined

## 6. Training the student

The student is a fresh LoRA on Minerva-7B-instruct, trained on the generated tasks only. Each task becomes a prompt/completion pair in the same format used for evaluation in Section 7:

```
{passage}

{instruction} {question} Vero o Falso?
Risposta:
```

with completion ` Vero` or ` Falso`. The loss is computed on the completion only, 5% of the training data is held out to pick the best checkpoint.

The next cell splits the rows by passage, loads a new 4-bit base model, adds a LoRA and trains it with the shared arguments of Section 1.

> **Note:** this section is **not meant to be re-run**. Training the student takes several GPU-hours: to train it again, use the script `scripts/train_student.py`. The run is documented here for transparency and as a reference. The trained adapter is provided in `ckpt/student_bonita/`, from where Section 7 loads it: the next cell can be skipped.

In [ ]:
# the generated tasks are reloaded from disk if they exist, so Sections 3-5 can be skipped
if os.path.exists(GENERATED_PATH):
    print(f"loading the generated tasks from {GENERATED_PATH}")
    generated = read_jsonl(GENERATED_PATH)

student_rows = [
    {
        "prompt": PROMPT_TEMPLATE.format(passage=r["passage"], task=r["task"]),
        "completion": " " + answer_text(r["label"]),
        "passage": r["passage"],
    }
    for r in generated
]
student_train, student_dev = split_by_group(student_rows, "passage", frac=DEV_FRAC, seed=SEED)
print(f"student train: {len(student_train)}, dev: {len(student_dev)}")
print(student_train[0]["prompt"] + student_train[0]["completion"])

tokenizer = AutoTokenizer.from_pretrained(INSTRUCTED_MODEL)
train_lora(load_model(model_name=INSTRUCTED_MODEL), tokenizer, student_train, student_dev, STUDENT_DIR, STUDENT_EPOCHS)
free_memory()

## 7. Log-likelihood evaluation on the held-out 20%

Every test example is turned into the prompt of Section 6, with a fixed instruction (the first variant):

```
{passage}

Dopo aver letto il passaggio, rispondi alla seguente domanda: {question} Vero o Falso?
Risposta:
```

and the model scores the two continuations ` Vero` and ` Falso`: $\log P(\text{continuation} \mid \text{prompt})$ is the sum of the log-probabilities of the continuation tokens. A continuation can span several tokens (` Vero` is one token, ` Falso` two: `ĠFal` + `so`), and the probability of each token after the first depends on the previous ones: each (prompt, continuation) pair is therefore scored with its own forward pass, i.e. two forward passes per example, one for each continuation. The answer is the continuation with the highest score. We report the accuracy and the macro-F1, since the classes are unbalanced (61% `Vero`). The baseline that always answers `Vero` (majority class) is shown for reference.

In [15]:
CHOICES = [" Falso", " Vero"]  # index = gold label (False = 0, True = 1)

# one test set per instruction variant: same questions and labels, different instruction
eval_prompts = [
    [
        PROMPT_TEMPLATE.format(
            passage=ex["metadata"]["passage_translation"],
            task=f"{instruction} {ex['input_translation']} {QUESTION_SUFFIX}",
        )
        for ex in boolq_test
    ]
    for instruction in INSTRUCTION_VARIANTS
]
gold_labels = np.array([int(ex["label"]) for ex in boolq_test])
print(len(gold_labels), "test examples,", len(eval_prompts), "instruction variants")
print(eval_prompts[0][0])


def encode_request(tokenizer, context: str, continuation: str) -> tuple[list[int], int]:
    """Returns the token ids of context + continuation and the number of context tokens."""
    context_ids = tokenizer(context).input_ids
    return context_ids + tokenizer(continuation, add_special_tokens=False).input_ids, len(context_ids)


@torch.no_grad()
def continuation_logprobs(
    model, tokenizer, contexts: list[str], continuations: list[str], batch_size: int = 16
) -> np.ndarray:
    """Sum of the log-probabilities of each continuation given its context."""
    requests = [encode_request(tokenizer, c, x) for c, x in zip(contexts, continuations, strict=True)]
    scores = []
    for start in tqdm(range(0, len(requests), batch_size)):
        batch = requests[start : start + batch_size]
        # right padding: the positions of the real tokens are unchanged
        inputs = tokenizer.pad({"input_ids": [ids for ids, _ in batch]}, padding_side="right", return_tensors="pt")
        logits = model(**inputs.to(model.device)).logits
        for row, (ids, n_context) in enumerate(batch):
            # the token at position t is predicted by the logits at position t - 1
            logprobs = torch.log_softmax(logits[row, n_context - 1 : len(ids) - 1].float(), dim=-1)
            targets = torch.tensor(ids[n_context:], device=logprobs.device)
            scores.append(logprobs.gather(-1, targets[:, None]).sum().item())
    return np.array(scores)


def evaluate_model(model, tokenizer, prompts: list[str]) -> np.ndarray:
    """Predicted labels: the index of the continuation with the highest log-likelihood."""
    contexts = [p for p in prompts for _ in CHOICES]
    continuations = [c for _ in prompts for c in CHOICES]
    return continuation_logprobs(model, tokenizer, contexts, continuations).reshape(-1, len(CHOICES)).argmax(axis=1)

2490 test examples, 3 instruction variants
Now You See Me è una serie di film di genere thriller e colpi scritti da Ed Solomon, Boaz Yakin e Edward Ricourt. I film si concentrano sulle azioni di un team di illusionisti chiamati ``The Four Horsemen'' che portano a termine colpi quasi impossibili. La serie presenta un cast di attori tra cui Jesse Eisenberg, Mark Ruffalo, Woody Harrelson, Isla Fisher, Dave Franco, Michael Caine, Lizzy Caplan e Morgan Freeman. Il primo film è stato rilasciato nel 2013, mentre il secondo è stato rilasciato nel 2016 e un terzo film è attualmente in fase di sviluppo e dovrebbe essere rilasciato nel 2019. La serie ha ricevuto recensioni miste da critici e pubblico e ha incassato quasi 700 milioni di dollari in tutto il mondo.

Dopo aver letto il passaggio, rispondi alla seguente domanda: Ci sarà un terzo capitolo di Now You See Me? Vero o Falso?
Risposta:


Minerva-7B-instruct is loaded once in 4-bit with the student's adapter: the student is scored with the adapter on, the baseline with the adapter turned off.

In [16]:
tokenizer = AutoTokenizer.from_pretrained(INSTRUCTED_MODEL)
model = PeftModel.from_pretrained(load_model(model_name=INSTRUCTED_MODEL), STUDENT_DIR)
model.eval()

# one array of predicted labels per instruction variant
predictions: dict[str, list[np.ndarray]] = {"Baseline": [], "Student": []}
for prompts in eval_prompts:
    with model.disable_adapter():
        predictions["Baseline"].append(evaluate_model(model, tokenizer, prompts))
    predictions["Student"].append(evaluate_model(model, tokenizer, prompts))

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

  0%|          | 0/312 [00:00<?, ?it/s]

: 

## 8. Results

For each model, accuracy and macro-F1 are computed from the predictions of Section 7. The majority baseline answers `Vero` to every question: its accuracy is the share of `Vero` in the test set, while its macro-F1 is low, because the `Falso` class is never predicted.

In [ ]:
def macro_f1(pred: np.ndarray, gold: np.ndarray) -> float:
    f1s = []
    for c in (0, 1):
        tp = np.sum((pred == c) & (gold == c))
        fp = np.sum((pred == c) & (gold != c))
        fn = np.sum((pred != c) & (gold == c))
        f1s.append(2 * tp / (2 * tp + fp + fn) if tp + fp + fn else 0.0)
    return float(np.mean(f1s))


def summarize(pred: np.ndarray) -> dict[str, float]:
    return {"acc": float(np.mean(pred == gold_labels)), "macro_f1": macro_f1(pred, gold_labels)}


results = {"always Vero (majority)": summarize(np.ones_like(gold_labels))}
for name, preds in predictions.items():
    per_variant = [summarize(pred) for pred in preds]
    for i, scores in enumerate(per_variant):
        results[f"{name}, instruction {i}"] = scores
    results[f"{name}, mean"] = {k: float(np.mean([s[k] for s in per_variant])) for k in per_variant[0]}

results_table = pd.DataFrame(results).T
display(results_table)

,acc,macro_f1
always Vero (majority),0.609639,0.378743
Minerva-7B-instruct (base),0.797189,0.794341
student (BONITA data),0.884739,0.877809


In [9]:
saved = {}
if os.path.exists(RESULTS_PATH):
    with open(RESULTS_PATH) as f:
        saved = json.load(f)
with open(RESULTS_PATH, "w") as f:
    json.dump(
        {
            "generation": counts if "counts" in globals() else saved.get("generation"),
            "results": results,
        },
        f,
        indent=1,
    )